# Operate a library assistant rollout

An offline fixture represents a library assistant rollout. Build trace, cost, permission and release checks using synthetic data. All exercises run on a laptop CPU.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.


## 1. Project safe trace fields

Project logs to selected non-content fields.

Implement trace_metadata(event). Input fields have already been type-validated. Keep only trace_id, operation, status, duration_ms, input_tokens and output_tokens. Do not invent missing fields. Preserve zero-valued counts.


In [ ]:
def trace_metadata(event):
    raise NotImplementedError


In [ ]:
assert trace_metadata({"trace_id":"a","prompt":"fixture","duration_ms":0,"unknown":3}) == {"trace_id":"a","duration_ms":0}
assert trace_metadata({"output_tokens":0,"input_tokens":5,"status":"error","completion":"fixture"}) == {"status":"error","input_tokens":5,"output_tokens":0}
assert trace_metadata({}) == {}
print("OK")


## 2. Estimate a token bill

Calculate a request cost estimate.

Implement token_cost for nonnegative counts and USD prices per million tokens. Return the USD estimate. The fixture ignores cache discounts and per-tool fees.


In [ ]:
def token_cost(input_tokens, output_tokens, input_per_million, output_per_million):
    raise NotImplementedError


In [ ]:
assert abs(token_cost(1000,100,2,8) - 0.0028) < 1e-12
assert token_cost(0,1_000_000,2,8) == 8
assert token_cost(1_000_000,0,2,8) == 2
assert token_cost(0,0,2,8) == 0
print("OK")


## 3. Authorize a read-only tool

Enforce scoped read permissions.

Implement may_read(tool, requested_collection, trusted_collection, scopes). Allow catalogue_search only with catalogue:read and hours_lookup only with hours:read. Also require the requested collection to equal the trusted collection. Deny unknown tools.


In [ ]:
def may_read(tool, requested_collection, trusted_collection, scopes):
    raise NotImplementedError


In [ ]:
assert may_read("catalogue_search","a","a",{"catalogue:read"})
assert may_read("hours_lookup","a","a",{"hours:read"})
assert not may_read("catalogue_search","b","a",{"catalogue:read"})
assert not may_read("hours_lookup","a","a",{"catalogue:read"})
assert not may_read("unknown","a","a",{"catalogue:read","hours:read"})
print("OK")


## 4. Apply a canary gate

Apply a canary gate using paired thresholds.

Implement rollout_decision for valid counts and thresholds. A critical event rolls back immediately. Otherwise hold on zero attempts or insufficient sample. With enough attempts, roll back above the maximum error rate and promote at or below it.


In [ ]:
def rollout_decision(attempts, failed, critical, min_attempts, max_error):
    raise NotImplementedError


In [ ]:
assert rollout_decision(2,0,True,100,0.02) == "rollback"
assert rollout_decision(40,0,False,100,0.02) == "hold"
assert rollout_decision(0,0,False,0,0.02) == "hold"
assert rollout_decision(100,2,False,100,0.02) == "promote"
assert rollout_decision(100,3,False,100,0.02) == "rollback"
print("OK")


## 5. Combine redacted traces with a canary decision

Implement canary_report(events, min_attempts, max_error, input_price, output_price). Each event
is one completed attempt with status, input_tokens and output_tokens. Status ok is a success.
Status critical represents a critical incident, and all non-ok statuses count as failures.
Return (projected traces, total token cost, rollout decision). Use trace_metadata, token_cost and rollout_decision.
Sum costs over every attempted request, including failures. Do not include raw content.


In [ ]:
def canary_report(events, min_attempts, max_error, input_price, output_price):
    raise NotImplementedError


In [ ]:
events = [{"status":"ok","input_tokens":1000,"output_tokens":100,"prompt":"fixture"},
          {"status":"timeout","input_tokens":500,"output_tokens":0}]
traces, cost, decision = canary_report(events, 2, 0.1, 2, 8)
assert len(traces) == 2 and "prompt" not in traces[0]
assert abs(cost - 0.0038) < 1e-12 and decision == "rollback"
assert canary_report([], 5, 0.1, 2, 8) == ([],0,"hold")
assert canary_report([{"status":"critical","input_tokens":0,"output_tokens":0}], 5, 0.1, 2, 8)[2] == "rollback"
ok = {"status":"ok","input_tokens":0,"output_tokens":0}
bad = {"status":"error","input_tokens":0,"output_tokens":0}
assert canary_report([ok], 5, 0.1, 2, 8)[2] == "hold", "Hold below min_attempts"
assert canary_report([ok]*9 + [bad], 10, 0.1, 2, 8)[2] == "promote", "An error rate equal to max_error promotes"
assert canary_report([ok]*8 + [bad]*2, 10, 0.1, 2, 8)[2] == "rollback", "Roll back above max_error"
print("OK")


## Interpretation

Write an incident note for a canary with a rising error rate and a single denied access attempt. Give the denominator, decision, rollback target and two trace fields that help diagnose the event.

**Your answer**
